基于遗传算法和非线性规划的函数寻优算法
所谓的非线性优化，就是研究一个n元实函数在一组等式或不等式的约束条件下的极值问题。经典非线性规划算法大多采用梯度下降法，局部搜索能力强，但是全局搜索能力弱。
相对应的，遗传算法的全局搜索能力强而局部搜索能力弱，二者可以互补。

算法的大概思路是利用遗传算法，每进行N次迭代，用一次非线性寻优，每次遗传算法结束后判断是否符合终止条件。

遗传算法的实现
第一步 种群初始化，采用实数编码，相较于二进制编码，避免了编码解码时的误差w_1 = low_range + p * (high_range-low_range)
第二步 适应度函数依然是取最大值
第三步 选择操作选用轮盘赌法。个体i被选中的概率为适应度值/所有个体的适应度值总和
第四步 交叉，第k和l个染色体a_k和a_l在第j位的交叉方法为：a_kj=a_ij*(1-b)+a_lj*b 以及 a_lj=a_lj*(1-b)+a_kj*b 其中b=[0,1]，如果激进一些，上下限可以适当放开
第五步 变异，要求是迭代次数越大，变异幅度越小。对于地i个个体的第j个基因
a_ij = a_ij+(a_ij-a_max)*r_2*(1-g/G_max)^2 r>=0.5
a_ij = a_ij+(a_min-a_ij)*r_2*(1-g/G_max)^2 r<0.5
其中a_max，a_min是基因a_ij的上下界，r_2是[0,1]之间的随机数，g是当前迭代次数，G_max是最大进化次数，r为[0,1]之间的随机数

本例子对以下五元函数进行优化(极小)
f(x) = -5sin(x_1)sin(x_2)sin(x_3)sin(x_4)sin(x_5) - sin(5*x_1)*sin(5*x_2)*sin(5*x_3)*sin(5*x_4)*sin(5*x_5) + 8
约束条件：x_i=[0, 0.9pi]
理论最优值（最小）：f(x)=2     x=(0.5pi,0.5pi,0.5pi,0.5pi,0.5pi)

所面对的问题是优化一个五维函数
所有最小值问题都可以通过加一个负号等价于求最大值问题

In [30]:
# 生成种群
# 输入：个体数量，个体大小，个体的上下限
# 输出：种群
import numpy as np

def create_group(Nind, Lind, ranges):
    Chrome = np.zeros((Nind,Lind), dtype=np.float32)
    for i in range(Nind):
        p = np.random.rand(Lind)
        Chrome[i,:] = ranges[0,:] + np.multiply((ranges[1,:]-ranges[0,:]),p)
    return Chrome

In [33]:
# 适应度函数
# 输入：种群，适应度函数
# 输出：适应度value
# f(x) = -5sin(x_1)sin(x_2)sin(x_3)sin(x_4)sin(x_5) - sin(5*x_1)*sin(5*x_2)*sin(5*x_3)*sin(5*x_4)*sin(5*x_5) + 8
def value_function(x):
    return -5*np.sin(x[0])*np.sin(x[1])*np.sin(x[2])*np.sin(x[3])*np.sin(x[4]) - np.sin(5*x[0])*np.sin(5*x[1])*np.sin(5*x[2])*np.sin(5*x[3])*np.sin(5*x[4]) + 8

def get_value(Chrom, value_fuction):
    values = np.zeros((Chrom.shape[0],1))
    for i in range(Chrom.shape[0]):
        values[i,:] = value_fuction(Chrom[i,:])
    return values   # 求最小值时写为return -values

In [32]:
# 选择操作，轮盘赌法
# 输入：父代种群
# 输出：优良的父代种群
def select(father_chrome):
    fitness = get_value(father_chrome, value_function)
    # 使用softmax函数将fitness化为[0,1]之内的概率
    fitness_exp = np.exp(fitness)
    fitness_exp_sum = np.sum(fitness_exp)
    fitness_softmax = fitness_exp / fitness_exp_sum
    
    # 轮盘赌法：计算累积概率
    cumsum_prob = np.cumsum(fitness_softmax, axis=0)
    
    # 根据概率进行选择，选择与父代相同数量的个体
    Nind = father_chrome.shape[0]
    selected_chrome = np.zeros_like(father_chrome)
    
    for i in range(Nind):
        # 生成随机数[0,1]
        r = np.random.rand()
        # 找到第一个累积概率大于r的个体
        index = np.where(cumsum_prob >= r)[0][0]
        selected_chrome[i, :] = father_chrome[index, :]
    
    return selected_chrome

In [ ]:
# 交叉操作
# 第k和l个染色体a_k和a_l在第j位的交叉方法为：a_kj=a_ij*(1-b)+a_lj*b 以及 a_lj=a_lj*(1-b)+a_kj*b 其中b=[0,1]
# 输入：选择后的优良父代种群
# 输出：交叉后的结果
def cross(selected_chrome, p_cross):
    Nind, Lind = selected_chrome.shape
    decide_vector = np.random.rand(Nind)
    indices = np.where(decide_vector < p_cross)[0]
    if indices.size == 0:
        return selected_chrome  # 如果没有个体满足交叉条件，返回原种群
    selected_chrome = selected_chrome[indices, :]
    crossed_chrome = np.zeros((0,Lind))
    # 对于n个染色体，进行n-1次交叉
    for i in range(selected_chrome.shape[0]-1):
        if i==Nind-1:
            break
        else:
            a_k = selected_chrome[i,:]
            a_l = selected_chrome[i+1,:]
            b = np.random.rand()
            a_final = a_k * (1-b) + a_l * b
            crossed_chrome = np.vstack((crossed_chrome, a_final))
    return crossed_chrome

In [ ]:
# 变异，要求是迭代次数越大，变异幅度越小。
# 对于地i个个体的第j个基因
# a_ij = a_ij+(a_ij-a_max)*r_2*(1-g/G_max)^2 r>=0.5
# a_ij = a_ij+(a_min-a_ij)*r_2*(1-g/G_max)^2 r<0.5
# 其中a_max，a_min是基因a_ij的上下界，r_2是[0,1]之间的随机数
# g是当前迭代次数，G_max是最大进化次数，r为[0,1]之间的随机数
# 输入：交叉后的种群，变异率，当前迭代次数，最大迭代次数
# 输出：变异后的种群
def mutation(crossed_chrome, mutation_rate, g, g_max):
    Nind, Lind = crossed_chrome.shape
    mutated_chrome = np.copy(crossed_chrome)
    for i in range(Nind):
        for j in range(Lind):
            # 判断是否进行变异
            r = np.random.rand()
            if r < mutation_rate:
                r_2 = np.random.rand()
                if r_2 >= 0.5:
                    a_max = 0.9*np.pi  # 基因的上界
                    mutated_chrome[i,j] = mutated_chrome[i,j] + (a_max - mutated_chrome[i,j]) * r_2 * (1 - g / g_max) ** 2
                else:
                    a_min = 0  # 基因的下界
                    mutated_chrome[i,j] = mutated_chrome[i,j] + (a_min - mutated_chrome[i,j]) * r_2 * (1 - g / g_max) ** 2
    return mutated_chrome

In [ ]:
# 更新种群
# 输入：父代种群，变异后的种群
# 输出：新一代种群
def update_group(father_chrome, mutated_chrome):
    Nind_father = father_chrome.shape[0]
    new_chrome = np.vstack((father_chrome, mutated_chrome))
    new_value = get_value(new_chrome, value_function)
    sort_indices = np.argsort(-new_value.flatten())  # 降序排序
    new_chrome = new_chrome[sort_indices[:Nind_father], :]
    return new_chrome

In [ ]:
# 得到种群最优值
def get_best_individual(chrome, value_function):
    values = np.zeros((chrome.shape[0], 1))
    values = get_value(chrome, value_function)
    best_index = np.argmax(values)
    return chrome[best_index, :], values[best_index, 0]    # 最小值时写为return -values[best_index, 0]

In [28]:
# 非线性寻优
# 输入：一代种群，学习率
# 输出：新一代种群
def compute_gradient(func, x, h=1e-6):
    n = len(x)
    grad = np.zeros(n)
    for i in range(n):
        # 创建扰动向量
        x_plus = x.copy()
        x_minus = x.copy()
        # 在第i个维度上添加正负扰动
        x_plus[i] += h
        x_minus[i] -= h
        # 中心差分计算梯度
        grad[i] = (func(x_plus) - func(x_minus)) / (2 * h)
    return grad

def gradient_descent(func, chrom, learning_rate=0.001, max_iter=1000, tol=1e-6):
    for i in range(chrom.shape[0]):
        x = chrom[i, :].copy()
        for j in range(max_iter):
            # 计算梯度
            grad = compute_gradient(func, x)
            grad_norm = np.linalg.norm(grad)
            # 检查收敛
            if grad_norm < tol:
                print(f"在迭代 {j} 收敛")
                break
            # 更新参数
            x = x + learning_rate * grad
            if x.max() > 0.9 * np.pi or x.min() < 0:
                x = x - learning_rate * grad
                break
        chrom[i, :] = x
    return chrom

In [29]:
# 开始计算
Nind, size = 100, 5
ranges = np.array([[0, 0, 0, 0, 0], [0.9*np.pi, 0.9*np.pi, 0.9*np.pi, 0.9*np.pi, 0.9*np.pi]])  # 基因的上下界
Maxgen = 40
cross_rate = 0.7
mutate_rate = 0.1
result = np.zeros((Maxgen,1), dtype=np.float32)  # 用于存储每代的最优值
# 生成种群(Nind*Lind)
Chrom = create_group(Nind, size, ranges)

# 用于存储所有代数的数据（用于绘图）
all_generations = []
all_best_values = []
all_best_x = []

for gen in range(Maxgen):
    # 选择父代
    selected_father = select(Chrom)
    # 交叉
    crossed_Chrom = cross(selected_father, cross_rate)
    # 变异
    mutated_Chrom = mutation(crossed_Chrom, mutate_rate, gen, Maxgen)
    # 更新种群
    Chrom = update_group(Chrom, mutated_Chrom)
    # 输出当前代数和最优个体
    x, best_individual = get_best_individual(Chrom, value_function)
    result[gen,:] = [best_individual]
    all_generations.append(gen + 1)
    all_best_values.append(best_individual)
    all_best_x.append(x)
    print(f"Generation {gen+1}: Best Individual = {best_individual}:Best x {x}")
    if gen % 10 == 9:
        # 使用非线性优化方法对当前最优个体进行优化
        Chrom = gradient_descent(value_function, Chrom, learning_rate=0.001, max_iter=100, tol=1e-6)
        # 更新种群中的最优个体
        x, best_individual = get_best_individual(Chrom, value_function)
        result[gen,:] = [best_individual]
        print(f"Generation {gen+1}: Best Individual = {best_individual}:Best x {x}")
    # 如果数据已经连续5代没有优化，则取中断循环
    if gen > 4 and np.argmax(result[:,0]) < gen - 10:
        print("No improvement in the last 5 generations, stopping early.")
        break
# 输出最终结果
best_value = result[np.argmax(result[:,0]), :]
print(f"Best Individual = {best_value}")

Generation 1: Best Individual = 8.401586570660522:Best x [0.27822131 2.08866286 0.41720936 2.73000431 1.64986825]
Generation 2: Best Individual = 8.430259257213978:Best x [2.75391286 1.50749566 0.25046624 1.51478532 0.94716076]
Generation 3: Best Individual = 8.430259257213978:Best x [2.75391286 1.50749566 0.25046624 1.51478532 0.94716076]
Generation 4: Best Individual = 8.430259257213978:Best x [2.75391286 1.50749566 0.25046624 1.51478532 0.94716076]
Generation 5: Best Individual = 8.430259257213978:Best x [2.75391286 1.50749566 0.25046624 1.51478532 0.94716076]
Generation 6: Best Individual = 8.430259257213978:Best x [2.75391286 1.50749566 0.25046624 1.51478532 0.94716076]
Generation 7: Best Individual = 8.430259257213978:Best x [2.75391286 1.50749566 0.25046624 1.51478532 0.94716076]
Generation 8: Best Individual = 8.430259257213978:Best x [2.75391286 1.50749566 0.25046624 1.51478532 0.94716076]
Generation 9: Best Individual = 8.430259257213978:Best x [2.75391286 1.50749566 0.250466